# Workshop Databricks — Genie + AI/BI
## Guia passo a passo


| Módulo | Resultado |
|---|---|
| Criar tabelas com SQL | Quatro tabelas no Unity Catalog |
| Genie best practices | Agent pequeno e bem curado |
| Laboratório Genie | Perguntas e SQL revisados |
| AI/BI com Genie Code | Dashboard com gráficos e mapas |
| Vega-Lite | Custom Viz de disponibilidade |
| Importar BI (opcional) | Visão do fluxo `/importBI` |
| Recap | Checklist de qualidade |

## Modelo de dados

O workshop usa **exatamente quatro tabelas**. Todas as tabelas de fatos se ligam a `ativos_geo` por `ativo_id`.

```text
ativos_geo (ativo_id)
   ├──< medicoes_horarias (ativo_id)
   ├──< eventos_operacionais (ativo_id)
   └──< rotas_inspecao (ativo_id)
```
---

### `ativos_geo`
Cadastro de ativos de geração, com capacidade e localização.

| Coluna | Tipo | Descrição |
|---|---|---|
| `ativo_id` | STRING | Identificador único do ativo (ex.: `ATV-001`). Chave primária. |
| `ativo_nome` | STRING | Nome fictício do ativo (parque ou usina). |
| `tipo_geracao` | STRING | Tecnologia: `Solar`, `Eólica` ou `Hídrica`. |
| `regiao` | STRING | Macroregião: `Norte`, `Nordeste`, `Centro-Oeste`, `Sudeste` ou `Sul`. |
| `estado_nome` | STRING | Nome do estado. |
| `estado_sigla` | STRING | UF do estado (ex.: `SP`, `BA`). |
| `pais` | STRING | País da localidade. |
| `latitude` | DOUBLE | Latitude WGS84 em graus decimais. |
| `longitude` | DOUBLE | Longitude WGS84 em graus decimais. |
| `capacidade_mw` | DOUBLE | Capacidade nominal instalada, em **MW**. |
| `data_comissionamento` | DATE | Data em que o ativo entrou em operação. |

---

### `medicoes_horarias`
Medições por ativo e hora: geração, carga e disponibilidade.

| Coluna | Tipo | Descrição |
|---|---|---|
| `medicao_id` | STRING | Identificador único da medição. |
| `ativo_id` | STRING | Referência ao ativo (`ativos_geo.ativo_id`). |
| `medicao_ts` | TIMESTAMP | Data e hora da medição (granularidade horária). |
| `geracao_mwh` | DOUBLE | Energia gerada na hora, em **MWh**. |
| `carga_mw` | DOUBLE | Carga observada na hora, em **MW**. |
| `disponibilidade_pct` | DOUBLE | Disponibilidade operacional de **0 a 100**. Nunca some este campo; use média. |
| `temperatura_c` | DOUBLE | Temperatura ambiente no ativo, em °C. |
| `qualidade_dado` | STRING | Origem da medição: `Medido` ou `Estimado`. |

---

### `eventos_operacionais`
Eventos de manutenção, falha e restrição, com impacto estimado.

| Coluna | Tipo | Descrição |
|---|---|---|
| `evento_id` | STRING | Identificador único do evento. |
| `ativo_id` | STRING | Referência ao ativo (`ativos_geo.ativo_id`). |
| `inicio_ts` | TIMESTAMP | Início do evento. |
| `fim_ts` | TIMESTAMP | Fim do evento. Duração = `fim_ts - inicio_ts`. |
| `severidade` | STRING | Gravidade: `Baixa`, `Média`, `Alta` ou `Crítica`. |
| `categoria_evento` | STRING | Tipo: `Manutenção planejada`, `Falha de comunicação`, `Proteção acionada`, `Inspeção preventiva` ou `Restrição de rede`. |
| `planejado` | BOOLEAN | `true` se o evento estava no plano de manutenção; `false` se não planejado. |
| `energia_nao_suprida_mwh` | DOUBLE | Estimativa de energia não suprida, em **MWh**. |
| `status_evento` | STRING | Situação: `Aberto` ou `Encerrado`. |

---

### `rotas_inspecao`
Pontos ordenados de rotas de inspeção.

| Coluna | Tipo | Descrição |
|---|---|---|
| `ponto_id` | STRING | Identificador único do ponto na rota. |
| `rota_id` | STRING | Identificador do trajeto. |
| `ordem_ponto` | INTEGER | Sequência crescente dos pontos dentro da rota. |
| `ponto_ts` | TIMESTAMP | Horário em que a equipe passou pelo ponto. |
| `ativo_id` | STRING | Ativo inspecionado (`ativos_geo.ativo_id`). |
| `equipe` | STRING | Equipe responsável: `Equipe Norte`, `Equipe Sul` ou `Equipe Leste`. |
| `latitude` | DOUBLE | Latitude WGS84 do ponto. |
| `longitude` | DOUBLE | Longitude WGS84 do ponto. |
| `distancia_acumulada_km` | DOUBLE | Distância acumulada desde o início da rota, em km. |
| `status_rota` | STRING | Situação do trajeto: `Concluída` ou `Em andamento`. |

---
## Configuração inicial

Preencha os widgets com o catálogo, o schema e um prefixo individual. Exemplo de prefixo: `carol`.

As tabelas serão criadas como `{catalogo}.{schema}.{prefixo}_ativos_geo`.

>

<div style="padding: 15px; border-left: 5px solid #d9534f; background-color: #fdf7f7; color: #a94442;">
    <strong>🛑 ATENÇÃO:</strong> Lembre de substituir o caminho do dado pelo caminho da sua pasta. 
    
    Exemplo /Workspace/Users/seu_usuario@seu_email.com/workshop-genie-aibi-energia/dados



In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("nome_catalogo", "workspace", "Catálogo")
dbutils.widgets.text("nome_schema", "workshop_energia", "Schema")
dbutils.widgets.text("seu_prefixo", "prefix_", "Seu prefixo")
dbutils.widgets.text(
    "caminho_dados",
    "/Workspace/Users/{seu_usuario}/workshop-genie-aibi-energia/dados",
    "Caminho da pasta dados",
)

nome_catalogo = dbutils.widgets.get("nome_catalogo").strip()
nome_schema = dbutils.widgets.get("nome_schema").strip()
seu_prefixo = dbutils.widgets.get("seu_prefixo").strip()
caminho_dados = dbutils.widgets.get("caminho_dados").strip().rstrip("/")

assert nome_catalogo, "Informe o catálogo"
assert nome_schema, "Informe o schema"
assert seu_prefixo, "Informe um prefixo individual"
assert caminho_dados, "Informe o caminho da pasta dados"
print(f"Destino: {nome_catalogo}.{nome_schema} | Prefixo: {seu_prefixo}")
print(f"Dados: {caminho_dados}")


---
## Módulo 1 — Criar tabelas com SQL

Execute as células abaixo na ordem. Elas usam os widgets (`${nome_catalogo}`, `${nome_schema}`, `${seu_prefixo}`, `${caminho_dados}`) nas queries.

Tabelas criadas:

| CSV | Tabela |
|---|---|
| `ativos_geo.csv` | `{seu_prefixo}_ativos_geo` |
| `medicoes_horarias.csv` | `{seu_prefixo}_medicoes_horarias` |
| `eventos_operacionais.csv` | `{seu_prefixo}_eventos_operacionais` |
| `rotas_inspecao.csv` | `{seu_prefixo}_rotas_inspecao` |

Se o `read_files` falhar, use a alternativa manual no final deste módulo.


In [0]:
%sql
CREATE OR REPLACE TABLE `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_ativos_geo` (
  ativo_id STRING COMMENT 'Identificador único do ativo',
  ativo_nome STRING COMMENT 'Nome fictício do ativo (parque ou usina)',
  tipo_geracao STRING COMMENT 'Tecnologia de geração: Solar, Eólica ou Hídrica',
  regiao STRING COMMENT 'Macroregião do ativo',
  estado_nome STRING COMMENT 'Nome do estado',
  estado_sigla STRING COMMENT 'UF do estado',
  pais STRING COMMENT 'País da localidade',
  latitude DOUBLE COMMENT 'Latitude WGS84 em graus decimais',
  longitude DOUBLE COMMENT 'Longitude WGS84 em graus decimais',
  capacidade_mw DOUBLE COMMENT 'Capacidade nominal instalada em MW',
  data_comissionamento DATE COMMENT 'Data em que o ativo entrou em operação'
)
COMMENT 'Cadastro de ativos de geração, com capacidade e localização';

INSERT OVERWRITE TABLE `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_ativos_geo`
SELECT
  CAST(ativo_id AS STRING) AS ativo_id,
  CAST(ativo_nome AS STRING) AS ativo_nome,
  CAST(tipo_geracao AS STRING) AS tipo_geracao,
  CAST(regiao AS STRING) AS regiao,
  CAST(estado_nome AS STRING) AS estado_nome,
  CAST(estado_sigla AS STRING) AS estado_sigla,
  CAST(pais AS STRING) AS pais,
  CAST(latitude AS DOUBLE) AS latitude,
  CAST(longitude AS DOUBLE) AS longitude,
  CAST(capacidade_mw AS DOUBLE) AS capacidade_mw,
  CAST(data_comissionamento AS DATE) AS data_comissionamento
FROM read_files(
  '${caminho_dados}/ativos_geo.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);


In [0]:
%sql
CREATE OR REPLACE TABLE `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_medicoes_horarias` (
  medicao_id STRING COMMENT 'Identificador único da medição',
  ativo_id STRING COMMENT 'Referência ao ativo em ativos_geo',
  medicao_ts TIMESTAMP COMMENT 'Data e hora da medição, granularidade horária',
  geracao_mwh DOUBLE COMMENT 'Energia gerada na hora, em MWh',
  carga_mw DOUBLE COMMENT 'Carga observada na hora, em MW',
  disponibilidade_pct DOUBLE COMMENT 'Disponibilidade operacional de 0 a 100. Usar média, nunca soma',
  temperatura_c DOUBLE COMMENT 'Temperatura ambiente no ativo, em Celsius',
  qualidade_dado STRING COMMENT 'Origem da medição: Medido ou Estimado'
)
COMMENT 'Medições por ativo e hora: geração, carga e disponibilidade';

INSERT OVERWRITE TABLE `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_medicoes_horarias`
SELECT
  CAST(medicao_id AS STRING) AS medicao_id,
  CAST(ativo_id AS STRING) AS ativo_id,
  CAST(medicao_ts AS TIMESTAMP) AS medicao_ts,
  CAST(geracao_mwh AS DOUBLE) AS geracao_mwh,
  CAST(carga_mw AS DOUBLE) AS carga_mw,
  CAST(disponibilidade_pct AS DOUBLE) AS disponibilidade_pct,
  CAST(temperatura_c AS DOUBLE) AS temperatura_c,
  CAST(qualidade_dado AS STRING) AS qualidade_dado
FROM read_files(
  '${caminho_dados}/medicoes_horarias.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);


In [0]:
%sql
CREATE OR REPLACE TABLE `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_eventos_operacionais` (
  evento_id STRING COMMENT 'Identificador único do evento',
  ativo_id STRING COMMENT 'Referência ao ativo em ativos_geo',
  inicio_ts TIMESTAMP COMMENT 'Início do evento',
  fim_ts TIMESTAMP COMMENT 'Fim do evento. Duração = fim_ts - inicio_ts',
  severidade STRING COMMENT 'Gravidade: Baixa, Média, Alta ou Crítica',
  categoria_evento STRING COMMENT 'Tipo do evento operacional',
  planejado BOOLEAN COMMENT 'true se estava no plano de manutenção',
  energia_nao_suprida_mwh DOUBLE COMMENT 'Estimativa de energia não suprida, em MWh',
  status_evento STRING COMMENT 'Situação: Aberto ou Encerrado'
)
COMMENT 'Eventos de manutenção, falha e restrição, com impacto estimado';

INSERT OVERWRITE TABLE `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_eventos_operacionais`
SELECT
  CAST(evento_id AS STRING) AS evento_id,
  CAST(ativo_id AS STRING) AS ativo_id,
  CAST(inicio_ts AS TIMESTAMP) AS inicio_ts,
  CAST(fim_ts AS TIMESTAMP) AS fim_ts,
  CAST(severidade AS STRING) AS severidade,
  CAST(categoria_evento AS STRING) AS categoria_evento,
  CAST(planejado AS BOOLEAN) AS planejado,
  CAST(energia_nao_suprida_mwh AS DOUBLE) AS energia_nao_suprida_mwh,
  CAST(status_evento AS STRING) AS status_evento
FROM read_files(
  '${caminho_dados}/eventos_operacionais.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);


In [0]:
%sql
CREATE OR REPLACE TABLE `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_rotas_inspecao` (
  ponto_id STRING COMMENT 'Identificador único do ponto na rota',
  rota_id STRING COMMENT 'Identificador do trajeto',
  ordem_ponto INT COMMENT 'Sequência crescente dos pontos dentro da rota',
  ponto_ts TIMESTAMP COMMENT 'Horário em que a equipe passou pelo ponto',
  ativo_id STRING COMMENT 'Ativo inspecionado, referência a ativos_geo',
  equipe STRING COMMENT 'Equipe responsável pela inspeção',
  latitude DOUBLE COMMENT 'Latitude WGS84 do ponto',
  longitude DOUBLE COMMENT 'Longitude WGS84 do ponto',
  distancia_acumulada_km DOUBLE COMMENT 'Distância acumulada desde o início da rota, em km',
  status_rota STRING COMMENT 'Situação do trajeto: Concluída ou Em andamento'
)
COMMENT 'Pontos ordenados de rotas de inspeção';

INSERT OVERWRITE TABLE `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_rotas_inspecao`
SELECT
  CAST(ponto_id AS STRING) AS ponto_id,
  CAST(rota_id AS STRING) AS rota_id,
  CAST(ordem_ponto AS INT) AS ordem_ponto,
  CAST(ponto_ts AS TIMESTAMP) AS ponto_ts,
  CAST(ativo_id AS STRING) AS ativo_id,
  CAST(equipe AS STRING) AS equipe,
  CAST(latitude AS DOUBLE) AS latitude,
  CAST(longitude AS DOUBLE) AS longitude,
  CAST(distancia_acumulada_km AS DOUBLE) AS distancia_acumulada_km,
  CAST(status_rota AS STRING) AS status_rota
FROM read_files(
  '${caminho_dados}/rotas_inspecao.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);


---
### Alternativa — upload manual dos CSVs

Use este caminho só se o SQL com `read_files` não funcionar.

1. Abra **+ New > Add or upload data** ou **Catalog > Create > Create table**.
2. Faça o download dos CSVs da pasta `dados/` e faça o upload de cada arquivo, um por um.
3. Revise os tipos detectados, principalmente timestamps, booleanos, latitude e longitude.
4. Selecione o mesmo catálogo e schema dos widgets.
5. Crie as tabelas com estes nomes. Não esqueça o **prefixo_**.

| Arquivo | Nome da tabela |
|---|---|
| `ativos_geo.csv` | `{seu_prefixo}_ativos_geo` |
| `medicoes_horarias.csv` | `{seu_prefixo}_medicoes_horarias` |
| `eventos_operacionais.csv` | `{seu_prefixo}_eventos_operacionais` |
| `rotas_inspecao.csv` | `{seu_prefixo}_rotas_inspecao` |


---
## Módulo 2 — Genie best practices

Trate o Genie como uma pessoa analista recém-chegada ao domínio:

1. **Comece pequeno:** quatro fontes são suficientes para este objetivo.
2. **Documente tabelas e colunas:** explicite granularidade e unidades.
3. **Prefira semântica estruturada:** metric views, expressões SQL e exemplos antes de instruções livres.
4. **Use exemplos SQL verificados:** ensine joins, períodos e cálculos ambíguos.
5. **Escreva instruções específicas:** gatilho, dado ausente, ação e pergunta de esclarecimento.
6. **Evite conflitos:** exemplos e instruções devem usar as mesmas unidades e regras.
7. **Teste o SQL gerado:** compare com consultas conhecidas antes de compartilhar.
8. **Itere:** acompanhe feedback e perguntas reais.

### Criar o Genie Agent

1. Abra **Genie Agents > New**.
2. Adicione somente as quatro tabelas anteriores.
3. Depois clique em **CREATE**.
4. Clique em **Configure** e modifique o nome da sala Genie a Descrição

> Nome: Seu Nome - Operação de Energia — Workshop

> Descrição: Assistente para análise da operação de ativos de geração de energia. Permite explorar geração e carga, disponibilidade operacional, capacidade instalada, eventos e energia não suprida, além de rotas de inspeção e distribuição geográfica dos ativos. Responde a comparações por período, estado, região, ativo e tipo de geração, apoiando a identificação de tendências, riscos e oportunidades de melhoria operacional.

### Instruções para o Genie Agent (campo *Instructions*)

O campo de instruções do Genie aceita Markdown. Cole e adapte o bloco abaixo — ele usa títulos, listas e `código` para ficar legível e fácil de manter.

````markdown
# Assistente de Operação de Energia

Você é um analista que responde perguntas sobre uma operação de energia relacionada a eventos, rotas, ativos, disponibilidade e geração de energia.

## Regras obritatóras
- Responda em português.
- Sempre cite o período e as unidades.
- Arredonde medidas a duas casas.
- Sempre tente usar gráficos para ajudarem a explicar a métrica.
- Quando tiver dúvida sobre o tipo de agregação, pergunte ao usuário

## Regras de negócio
- Disponibilidade: percentual de 0 a 100. Nunca some; use média.
- Geração: em MWh (`geracao_mwh`).
- Capacidade e carga: em MW (`capacidade_mw`, `carga_mw`).
- Energia não suprida: em MWh (`energia_nao_suprida_mwh`).
- Duração de evento: `fim_ts - inicio_ts`.
- Localização: use `estado_nome` e `país`.
- Trajetos: agrupe por `rota_id` e ordene por `ordem_ponto`.

## Quando pedir esclarecimento
- "produção" sozinho → pergunte: geração em MWh ou capacidade em MW?
- tendência sem período → pergunte: Qual período deseja analisar?
- "impacto" ambíguo → pergunte: duração do evento ou energia não suprida?

````

> Instruções não substituem bons comentários de coluna, exemplos SQL verificados nem uma camada semântica correta.

### Examples na sala Genie

Na tela do Genie Agent, abra a aba **Examples**. Cada exemplo tem:

1. **What question does this query answer?** — a pergunta em linguagem natural
2. **SQL** — a consulta que responde essa pergunta
3. **Parameters** — deixe vazio nestes exemplos
4. **Usage Guidance** — quando o Genie deve reutilizar este padrão
5. **Preview** para validar, depois **Save**

<div style="padding: 15px; border-left: 5px solid #d9534f; background-color: #fdf7f7; color: #a94442;">
<strong>🛑 ATENÇÃO:</strong> rode as células Python abaixo. Elas imprimem pergunta, SQL já preenchido e Usage Guidance para copiar. O texto da célula SQL <em>não</em> troca `${widget}` sozinho.
</div>


---
**Exemplo 1** — rode a célula abaixo e copie os três blocos para o Genie.


In [0]:
nome_catalogo = dbutils.widgets.get("nome_catalogo").strip()
nome_schema = dbutils.widgets.get("nome_schema").strip()
seu_prefixo = dbutils.widgets.get("seu_prefixo").strip()

pergunta_1 = "Quais estados geraram mais energia e qual a disponibilidade média de cada um?"
guidance_1 = (
    "Use quando a pergunta pedir geração ou disponibilidade por estado. "
    "Sempre faça join com ativos_geo por ativo_id. "
    "Nunca some disponibilidade_pct; use média."
)
sql_exemplo_1 = f"""
SELECT
  a.estado_nome,
  ROUND(SUM(m.geracao_mwh), 2) AS geracao_total_mwh,
  ROUND(AVG(m.disponibilidade_pct), 2) AS disponibilidade_media_pct
FROM `{nome_catalogo}`.`{nome_schema}`.`{seu_prefixo}_medicoes_horarias` m
JOIN `{nome_catalogo}`.`{nome_schema}`.`{seu_prefixo}_ativos_geo` a
  USING (ativo_id)
GROUP BY a.estado_nome
ORDER BY geracao_total_mwh DESC
""".strip()

print("=== What question does this query answer? ===")
print(pergunta_1)
print()
print("=== SQL ===")
print(sql_exemplo_1)
print()
print("=== Usage Guidance ===")
print(guidance_1)

display(spark.sql(sql_exemplo_1))


---
**Exemplo 2** — rode a célula abaixo e copie os três blocos para o Genie.


In [0]:
nome_catalogo = dbutils.widgets.get("nome_catalogo").strip()
nome_schema = dbutils.widgets.get("nome_schema").strip()
seu_prefixo = dbutils.widgets.get("seu_prefixo").strip()

pergunta_2 = "Quais ativos tiveram mais energia não suprida em eventos não planejados?"
guidance_2 = (
    "Use para ranking de impacto operacional. "
    "Filtre planejado = false quando a pergunta for sobre falha ou evento não planejado. "
    "Join com ativos_geo por ativo_id."
)
sql_exemplo_2 = f"""
SELECT
  a.ativo_nome,
  a.estado_nome,
  COUNT(*) AS eventos,
  ROUND(SUM(e.energia_nao_suprida_mwh), 2) AS energia_nao_suprida_mwh
FROM `{nome_catalogo}`.`{nome_schema}`.`{seu_prefixo}_eventos_operacionais` e
JOIN `{nome_catalogo}`.`{nome_schema}`.`{seu_prefixo}_ativos_geo` a
  USING (ativo_id)
WHERE e.planejado = false
GROUP BY a.ativo_nome, a.estado_nome
ORDER BY energia_nao_suprida_mwh DESC
""".strip()

print("=== What question does this query answer? ===")
print(pergunta_2)
print()
print("=== SQL ===")
print(sql_exemplo_2)
print()
print("=== Usage Guidance ===")
print(guidance_2)

display(spark.sql(sql_exemplo_2))


---
## Módulo 3 — Laboratório Genie

Perguntas para fazer no Genie. Inspecione o SQL antes de aceitar a resposta.

1. `Quais estados geraram mais energia?`
2. `Compare a disponibilidade média de Solar, Eólica e Hídrica.`
3. `Quais eventos críticos tiveram maior duração e energia não suprida?`
4. `Mostre as rotas concluídas e sua distância final.`
5. `Como está a produção?`

Na pergunta 5, o Agent deve pedir esclarecimento.

Perguntas hipotéticas:

6. `Se reduzirmos em 20% a duração média dos eventos críticos, qual seria o impacto estimado na energia não suprida?`
7. `Se a disponibilidade média de Solar cair 3 pontos percentuais, quais estados sofreriam mais em geração?`
8. `Se priorizarmos inspeções nos ativos eólicos do Nordeste, quais rotas deveríamos antecipar e por quê?`
9. `Se encerrarmos todos os eventos abertos nas próximas 24 horas, quanto de energia não suprida deixaria de se acumular com base no histórico?`
10. `Se concentrarmos a Equipe Leste nas rotas em andamento, qual o impacto operacional estimado?`


---
## Módulo 4 — AI/BI com Genie Code

1. Abra **Dashboards > Create dashboard**.
2. Abra Genie Code e selecione **Agent mode**.
3. Referencie as quatro tabelas com `@`.
4. Rode a célula Python abaixo e copie o prompt já preenchido. Ele já pede os gráficos e os três mapas (choropleth, point e path).

<div style="padding: 15px; border-left: 5px solid #d9534f; background-color: #fdf7f7; color: #a94442;">
<strong>🛑 ATENÇÃO:</strong> rode a célula para substituir catálogo, schema e prefixo. Sem isso o Genie Code não encontra as tabelas.
</div>


In [0]:
nome_catalogo = dbutils.widgets.get("nome_catalogo").strip()
nome_schema = dbutils.widgets.get("nome_schema").strip()
seu_prefixo = dbutils.widgets.get("seu_prefixo").strip()

def tabela(sufixo):
    return f"`{nome_catalogo}`.`{nome_schema}`.`{seu_prefixo}_{sufixo}`"

ativos = tabela("ativos_geo")
medicoes = tabela("medicoes_horarias")
eventos = tabela("eventos_operacionais")
rotas = tabela("rotas_inspecao")

prompt_dashboard = f"""
Crie um dashboard chamado Operação de Energia — Workshop usando as quatro tabelas selecionadas.
Antes de alterar o dashboard, mostre um plano curto.

Crie duas páginas com vários gráficos de tipos diferentes e três mapas na página geográfica.
Evite gráfico de pizza. Títulos em português, unidades nos rótulos e duas casas para percentuais.
Nos gráficos de disponibilidade, comece o eixo Y em 70 (não em 0), para a diferença entre tipos aparecer.

Página 1 — Visão executiva
Coloque uma faixa de KPIs no topo e, abaixo, uma grade de gráficos:
- KPI: geração total em MWh
- KPI: disponibilidade média (%)
- KPI: quantidade de eventos críticos
- KPI: energia não suprida total em MWh
- Gráfico de linha: geração total por dia (some geracao_mwh por data, não por timestamp horário)
- Gráfico de linha: perfil médio de geração por hora do dia (0 a 23), uma série por tipo_geracao — deve aparecer a curva do Solar
- Gráfico de área: geração diária e carga média diária (séries com formas diferentes)
- Gráfico de barras verticais: geração total por tipo de geração (Solar, Eólica, Hídrica)
- Gráfico de barras horizontais: disponibilidade média por tipo de geração (eixo a partir de 70)
- Gráfico combinado: barras de geração e linha de disponibilidade média por estado
- Tabela: eventos críticos com ativo, severidade, duração e energia não suprida

Página 2 — Operação geográfica
Comece pelos mapas, em destaque:
- Choropleth por estado: localidade = estado_nome (State or province), cor = geração total em MWh, tooltip com disponibilidade média. Escala sequencial azul. Título: Geração por estado (MWh). País = Brasil.
- Point map dos ativos: latitude e longitude de ativos_geo, tamanho por capacidade_mw, cor por tipo_geracao, tooltip com ativo, estado e capacidade.
- Path map chamado Rotas de inspeção: ligue latitude e longitude de rotas_inspecao separadamente para cada rota_id, respeitando ordem_ponto. Uma cor por equipe. Tooltip com rota, ativo, horário, distância e status.

Em seguida, os demais gráficos:
- Gráfico de barras: geração total por estado
- Gráfico de barras agrupadas (não 100% empilhado): quantidade de eventos por severidade em cada estado
- Gráfico de dispersão: capacidade_mw no eixo X e geração total no eixo Y, um ponto por ativo, cor por tipo_geracao
- Heatmap: disponibilidade média por estado e tipo de geração
- Gráfico de barras: distância final das rotas por equipe (máximo de distancia_acumulada_km por rota, depois média ou soma por equipe)
- Gráfico de linha: distancia_acumulada_km ao longo de ordem_ponto, uma série por equipe (não por rota_id)
- Tabela: ranking de ativos com estado, tipo, geração, disponibilidade e energia não suprida

Regras dos mapas:
- rota_id separa os trajetos
- ordem_ponto define a sequência
- não inverter latitude e longitude
- filtros não podem eliminar pontos intermediários da rota (isso quebra o path)

Adicione filtros globais de estado e tipo de geração.

Use as tabelas do catálogo {nome_catalogo} e do schema {nome_schema} com prefixo {seu_prefixo}_ :
- {ativos}
- {medicoes}
- {eventos}
- {rotas}
""".strip()

print(prompt_dashboard)


### Usando uma imagem de referência para estilizar
1. Tire print de algum website
2. Copie e cole a imagem no chat da Genie e peça:
 > Estilize o dashboard seguindo como referência cores da imagem em anexo.

### Publicando o dashboard e usando a Genie para fazer perguntas

O rascunho do dashboard é só para quem edita. Para outras pessoas perguntarem sobre os dados, publique e use o **Ask Genie** na versão publicada.

#### Publicar

1. No dashboard, clique em **Publish** no canto superior direito.
2. Escolha como os viewers acessam os dados:
   - **Share data permissions** (recomendado neste workshop): as queries rodam com as credenciais de quem publicou. Quem não tem permissão nas tabelas ainda consegue ver o dashboard.
   - **Individual data permissions**: cada viewer usa as próprias permissões no Unity Catalog.
3. Deixe **Enable Genie** ligado. O Databricks cria um Genie Agent companion a partir dos datasets e gráficos do dashboard.
4. Confirme **Publish**.

A versão publicada fica congelada até você publicar de novo. Continuar editando o rascunho não altera o que o viewer vê.

Para compartilhar: clique em **Share**, adicione usuários ou grupos e conceda pelo menos permissão de visualização.

#### Perguntar no Ask Genie (dentro do AI/BI)

1. Abra a **versão publicada** do dashboard (não o draft).
2. Clique em **Ask Genie**.
3. Opcional: clique em um gráfico antes de perguntar. O Genie usa aquele visual como contexto.
4. Faça perguntas em português, por exemplo:
   - `Quais estados geraram mais energia?`
   - `Compare a disponibilidade média de Solar, Eólica e Hídrica.`
   - `Quais eventos críticos tiveram maior duração e energia não suprida?`
   - `Como está a produção?` — o Agent deve pedir esclarecimento.
5. Inspecione o SQL gerado antes de aceitar a resposta.
6. Use os filtros do dashboard (estado, tipo de geração) e veja se o Genie respeita o recorte visível.

O Ask Genie do dashboard responde com base nos datasets daquele dashboard. Para perguntas mais livres, fora dos gráficos, use a sala Genie do Módulo 2 ou o Genie One do módulo seguinte.


---
## Módulo 5 — Genie One

1. Acesse o topo da página. Clique no botão com nove bolinhas 
2. Abra o **Genie One**
3. Faça as mesmas perguntas que no Genie Agent.



---
## Módulo 6 — Vega-Lite Custom Visualization (opcional)

1. Abra o dashboard do Módulo 4.
2. Abra o Genie Code em **Agent mode**.
4. Selecione o gráfico **Distância acumulada por equipe**
3. Rode a célula abaixo e copie o prompt.

<div style="padding: 15px; border-left: 5px solid #d9534f; background-color: #fdf7f7; color: #a94442;">
<strong>🛑 ATENÇÃO:</strong> rode a célula antes de copiar.
</div>


In [0]:
prompt_vega = (
    "Transforme esse gráfico em um de gráfico feito com custom visualization" 
)

print(prompt_vega)


Depois que o Genie Code concluir, confira se o widget é uma **Custom Visualization**.


---
## Módulo 7 — Importar BI com Genie Code (opcional)

1. Anexe o .pbit do workshop.
2. Abra Genie Code, inicie uma conversa e escolha **Import from a BI tool** ou digite `/importBI` e faça upload do .pbit.
2a. Anexe `.pbit`, `.twb`, `.twbx`, `.tds` ou `.tdsx`; para arquivos grandes, use um caminho em Volume.
3. Forneça também uma captura de tela de boa qualidade.
4. Mantenha a aba aberta enquanto o agente trabalha.
5. Revise datasets, relacionamentos, cálculos, filtros, números e layout.
6. Promova metric views locais finalizadas para o Unity Catalog. Você também pode pedir para recriar o dashboard usando SQL puro ao invés do metric view.

Exemplo com Volume:

```text
/importBI
@/Volumes/meu_catalogo/meu_schema/meu_volume/modelo_energia.pbit
```